In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
import os


In [8]:
# --- PART 1: SETUP AND CONFIGURATION ---

# ✅ Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEXT_MODEL_PATH = "../google_muril/best_hindi_muril_large"
TEXT_EMB_DIM = 1024  # MuRIL-large base model output dimension
IMAGE_EMB_DIM = 512   # CLIP ViT-B/32 output dimension
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4

print(f"Using device: {DEVICE}")

Using device: cuda


In [9]:
# ✅ Load text data
df_train = pd.read_csv("train_clean.csv")
df_val = pd.read_csv("val_clean.csv")
df_test = pd.read_csv("test_clean.csv")


In [10]:
# ✅ Load image embeddings and labels
train_img_embs = np.load("../image_train/train_clip_embeddings.npy")
val_img_embs = np.load("../image_train/val_clip_embeddings.npy")
test_img_embs = np.load("../image_train/test_clip_embeddings.npy")

In [11]:
train_labels = np.load("../image_train/train_labels.npy")
val_labels = np.load("../image_train/val_labels.npy")
test_labels = np.load("../image_train/test_labels.npy")

In [12]:
train_labels = np.load("../image_train/train_labels.npy")
val_labels = np.load("../image_train/val_labels.npy")
test_labels = np.load("../image_train/test_labels.npy")

print("Loaded all text data and image embeddings. ✅")

Loaded all text data and image embeddings. ✅


In [13]:
# --- PART 2: GENERATE TEXT EMBEDDINGS ---

print("Loading fine-tuned text model for embedding extraction...")
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
text_encoder = AutoModel.from_pretrained(TEXT_MODEL_PATH).to(DEVICE)
text_encoder.eval()

def get_text_embeddings(texts, batch_size=64):
    """Function to get [CLS] token embeddings for a list of texts."""
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Generating text embeddings"):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(DEVICE)
            outputs = text_encoder(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.extend(cls_embeddings)
    return np.array(all_embeddings)

train_text_embs = get_text_embeddings(df_train['text'].tolist())
val_text_embs = get_text_embeddings(df_val['text'].tolist())
test_text_embs = get_text_embeddings(df_test['text'].tolist())

print(f"Generated text embeddings: Train shape {train_text_embs.shape}, Val shape {val_text_embs.shape}")

Loading fine-tuned text model for embedding extraction...


Generating text embeddings: 100%|██████████████████████████████████████████████████████| 29/29 [01:13<00:00,  2.54s/it]

Generated text embeddings: Train shape (8400, 1024), Val shape (1800, 1024)


In [14]:
# --- PART 3: PYTORCH DATASET AND FUSION MODEL ---

class MultimodalDataset(Dataset):
    """Custom PyTorch Dataset for our multimodal data."""
    def __init__(self, text_embs, img_embs, labels):
        self.text_embs = torch.tensor(text_embs, dtype=torch.float32)
        self.img_embs = torch.tensor(img_embs, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'text_emb': self.text_embs[idx],
            'img_emb': self.img_embs[idx],
            'label': self.labels[idx]
        }

train_dataset = MultimodalDataset(train_text_embs, train_img_embs, train_labels)
val_dataset = MultimodalDataset(val_text_embs, val_img_embs, val_labels)
test_dataset = MultimodalDataset(test_text_embs, test_img_embs, test_labels)

# --- MODIFICATION: Optimizing DataLoader for GPU ---
# Use num_workers to load data in parallel and pin_memory for faster CPU-to-GPU transfer.
# Note: num_workers > 0 might not work in some interactive environments like Jupyter notebooks on Windows.
# In that case, set num_workers=0.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
# --- END MODIFICATION ---

class FusionModel(nn.Module):
    """A simple fusion model that concatenates text and image embeddings."""
    def __init__(self, text_dim, image_dim, hidden_dim=512, num_labels=3):
        super(FusionModel, self).__init__()
        self.fusion_layer = nn.Linear(text_dim + image_dim, hidden_dim)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_dim, num_labels)
        self.relu = nn.ReLU()

    def forward(self, text_emb, img_emb):
        combined_features = torch.cat((text_emb, img_emb), dim=1)
        x = self.fusion_layer(combined_features)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits

model = FusionModel(text_dim=TEXT_EMB_DIM, image_dim=IMAGE_EMB_DIM).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print("\nFusion model created: ✅")
print(model)



Fusion model created: ✅
FusionModel(
  (fusion_layer): Linear(in_features=1536, out_features=512, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=512, out_features=3, bias=True)
  (relu): ReLU()
)


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
import os

# --- Keep classes and function definitions in the global scope ---

class MultimodalDataset(Dataset):
    """Custom PyTorch Dataset for our multimodal data."""
    def __init__(self, text_embs, img_embs, labels):
        self.text_embs = torch.tensor(text_embs, dtype=torch.float32)
        self.img_embs = torch.tensor(img_embs, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'text_emb': self.text_embs[idx],
            'img_emb': self.img_embs[idx],
            'label': self.labels[idx]
        }

class FusionModel(nn.Module):
    """A simple fusion model that concatenates text and image embeddings."""
    def __init__(self, text_dim, image_dim, hidden_dim=512, num_labels=3):
        super(FusionModel, self).__init__()
        self.fusion_layer = nn.Linear(text_dim + image_dim, hidden_dim)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_dim, num_labels)
        self.relu = nn.ReLU()

    def forward(self, text_emb, img_emb):
        combined_features = torch.cat((text_emb, img_emb), dim=1)
        x = self.fusion_layer(combined_features)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits

def get_text_embeddings(texts, tokenizer, model, device, batch_size=64):
    """Function to get [CLS] token embeddings for a list of texts."""
    model.eval()
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Generating text embeddings"):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)
            outputs = model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.extend(cls_embeddings)
    return np.array(all_embeddings)

def evaluate(model, data_loader, device):
    """Evaluate the model on a given dataset."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            text_emb = batch['text_emb'].to(device)
            img_emb = batch['img_emb'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(text_emb, img_emb)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


# --- NEW: Main execution function to hold all logic ---
def main():
    """Main function to run the setup, training, and evaluation process."""
    # --- PART 1: SETUP AND CONFIGURATION ---
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    TEXT_MODEL_PATH = "../google_muril/best_hindi_muril_large"
    TEXT_EMB_DIM = 1024
    IMAGE_EMB_DIM = 512
    BATCH_SIZE = 32
    EPOCHS = 10
    LEARNING_RATE = 1e-4
    BEST_MODEL_PATH = "best_fusion_model.pth"

    print(f"Using device: {DEVICE}")

    # Load text data
    df_train = pd.read_csv("train_clean.csv")
    df_val = pd.read_csv("val_clean.csv")
    df_test = pd.read_csv("test_clean.csv")
    
    # Load image embeddings and labels
    train_img_embs = np.load("../image_train/train_clip_embeddings.npy")
    val_img_embs = np.load("../image_train/val_clip_embeddings.npy")
    test_img_embs = np.load("../image_train/test_clip_embeddings.npy")
    train_labels = np.load("../image_train/train_labels.npy")
    val_labels = np.load("../image_train/val_labels.npy")
    test_labels = np.load("../image_train/test_labels.npy")
    print("Loaded all text data and image embeddings. ✅")

    # --- PART 2: GENERATE TEXT EMBEDDINGS ---
    print("Loading fine-tuned text model for embedding extraction...")
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
    text_encoder = AutoModel.from_pretrained(TEXT_MODEL_PATH).to(DEVICE)

    train_text_embs = get_text_embeddings(df_train['text'].tolist(), tokenizer, text_encoder, DEVICE)
    val_text_embs = get_text_embeddings(df_val['text'].tolist(), tokenizer, text_encoder, DEVICE)
    test_text_embs = get_text_embeddings(df_test['text'].tolist(), tokenizer, text_encoder, DEVICE)
    print(f"Generated text embeddings: Train shape {train_text_embs.shape}, Val shape {val_text_embs.shape}")

    # --- PART 3: SETUP DATASETS AND MODELS ---
    train_dataset = MultimodalDataset(train_text_embs, train_img_embs, train_labels)
    val_dataset = MultimodalDataset(val_text_embs, val_img_embs, val_labels)
    test_dataset = MultimodalDataset(test_text_embs, test_img_embs, test_labels)

    # train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    # val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    # test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = FusionModel(text_dim=TEXT_EMB_DIM, image_dim=IMAGE_EMB_DIM).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    print("\nFusion model created: ✅")
    print(model)

    # --- PART 4: TRAINING AND EVALUATION LOOP ---
    best_val_f1 = 0.0
    print("\nStarting training... 🚀")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
            optimizer.zero_grad()
            
            text_emb = batch['text_emb'].to(DEVICE)
            img_emb = batch['img_emb'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            
            logits = model(text_emb, img_emb)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            
            loss.backward()
            optimizer.step()
        
        avg_train_loss = total_loss / len(train_loader)
        
        val_metrics = evaluate(model, val_loader, DEVICE)
        print(f"Epoch {epoch + 1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val F1: {val_metrics['f1']:.4f} | Val Acc: {val_metrics['accuracy']:.4f}")
        
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"🎉 New best model saved to {BEST_MODEL_PATH} with F1: {best_val_f1:.4f}")

    print("\nTraining finished. ✅")

    # --- FINAL EVALUATION ON TEST SET ---
    print("\nLoading best model and evaluating on the test set...")
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    test_metrics = evaluate(model, test_loader, DEVICE)

    print("\n--- Test Set Results ---")
    print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"F1-Score:  {test_metrics['f1']:.4f}")
    print(f"Precision: {test_metrics['precision']:.4f}")
    print(f"Recall:    {test_metrics['recall']:.4f}")
    print("------------------------")


# --- NEW: The __main__ guard to protect the script's entry point ---
if __name__ == '__main__':
    main()

Using device: cuda
Loaded all text data and image embeddings. ✅
Loading fine-tuned text model for embedding extraction...


Generating text embeddings: 100%|██████████████████████████████████████████████████████| 29/29 [00:40<00:00,  1.40s/it]


Generated text embeddings: Train shape (8400, 1024), Val shape (1800, 1024)

Fusion model created: ✅
FusionModel(
  (fusion_layer): Linear(in_features=1536, out_features=512, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=512, out_features=3, bias=True)
  (relu): ReLU()
)

Starting training... 🚀


Epoch 1/10: 100%|███████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 511.38it/s]


Epoch 1/10 | Train Loss: 1.0770 | Val F1: 0.4952 | Val Acc: 0.4941
🎉 New best model saved to best_fusion_model.pth with F1: 0.4952


...

Epoch 8/10 | Train Loss: 0.7981 | Val F1: 0.5339 | Val Acc: 0.5489


Epoch 9/10: 100%|███████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 756.67it/s]


Epoch 9/10 | Train Loss: 0.7767 | Val F1: 0.5336 | Val Acc: 0.5326


Epoch 10/10: 100%|██████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 772.83it/s]


Epoch 10/10 | Train Loss: 0.7550 | Val F1: 0.5490 | Val Acc: 0.5526
🎉 New best model saved to best_fusion_model.pth with F1: 0.5490

Training finished. ✅

Loading best model and evaluating on the test set...

--- Test Set Results ---
Accuracy:  0.5541
F1-Score:  0.5510
Precision: 0.5511
Recall:    0.5541
------------------------


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
import os

# --- Keep classes and function definitions in the global scope ---

class MultimodalDataset(Dataset):
    """Custom PyTorch Dataset for our multimodal data."""
    def __init__(self, text_embs, img_embs, labels):
        self.text_embs = torch.tensor(text_embs, dtype=torch.float32)
        self.img_embs = torch.tensor(img_embs, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'text_emb': self.text_embs[idx],
            'img_emb': self.img_embs[idx],
            'label': self.labels[idx]
        }

class FusionModel(nn.Module):
    """A simple fusion model that concatenates text and image embeddings."""
    def __init__(self, text_dim, image_dim, hidden_dim=512, num_labels=3):
        super(FusionModel, self).__init__()
        self.fusion_layer = nn.Linear(text_dim + image_dim, hidden_dim)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_dim, num_labels)
        self.relu = nn.ReLU()

    def forward(self, text_emb, img_emb):
        combined_features = torch.cat((text_emb, img_emb), dim=1)
        x = self.fusion_layer(combined_features)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits

def get_text_embeddings(texts, tokenizer, model, device, batch_size=64):
    """Function to get [CLS] token embeddings for a list of texts."""
    model.eval()
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Generating text embeddings"):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)
            outputs = model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.extend(cls_embeddings)
    return np.array(all_embeddings)

def evaluate(model, data_loader, device):
    """Evaluate the model on a given dataset."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            text_emb = batch['text_emb'].to(device)
            img_emb = batch['img_emb'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(text_emb, img_emb)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


# --- NEW: Main execution function to hold all logic ---
def main():
    """Main function to run the setup, training, and evaluation process."""
    # --- PART 1: SETUP AND CONFIGURATION ---
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    TEXT_MODEL_PATH = "../google_muril/best_hindi_muril_large"
    TEXT_EMB_DIM = 1024
    IMAGE_EMB_DIM = 512
    BATCH_SIZE = 32
    EPOCHS = 10
    LEARNING_RATE = 1e-4
    BEST_MODEL_PATH = "best_fusion_model.pth"

    print(f"Using device: {DEVICE}")

    # Load text data
    df_train = pd.read_csv("train_clean.csv")
    df_val = pd.read_csv("val_clean.csv")
    df_test = pd.read_csv("test_clean.csv")
    
    # Load image embeddings and labels
    train_img_embs = np.load("../image_train/train_clip_embeddings.npy")
    val_img_embs = np.load("../image_train/val_clip_embeddings.npy")
    test_img_embs = np.load("../image_train/test_clip_embeddings.npy")
    train_labels = np.load("../image_train/train_labels.npy")
    val_labels = np.load("../image_train/val_labels.npy")
    test_labels = np.load("../image_train/test_labels.npy")
    print("Loaded all text data and image embeddings. ✅")

    # --- PART 2: GENERATE TEXT EMBEDDINGS ---
    print("Loading fine-tuned text model for embedding extraction...")
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
    text_encoder = AutoModel.from_pretrained(TEXT_MODEL_PATH).to(DEVICE)

    train_text_embs = get_text_embeddings(df_train['text'].tolist(), tokenizer, text_encoder, DEVICE)
    val_text_embs = get_text_embeddings(df_val['text'].tolist(), tokenizer, text_encoder, DEVICE)
    test_text_embs = get_text_embeddings(df_test['text'].tolist(), tokenizer, text_encoder, DEVICE)
    print(f"Generated text embeddings: Train shape {train_text_embs.shape}, Val shape {val_text_embs.shape}")

    # --- PART 3: SETUP DATASETS AND MODELS ---
    train_dataset = MultimodalDataset(train_text_embs, train_img_embs, train_labels)
    val_dataset = MultimodalDataset(val_text_embs, val_img_embs, val_labels)
    test_dataset = MultimodalDataset(test_text_embs, test_img_embs, test_labels)

    # train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    # val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    # test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = FusionModel(text_dim=TEXT_EMB_DIM, image_dim=IMAGE_EMB_DIM).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    print("\nFusion model created: ✅")
    print(model)

    # --- PART 4: TRAINING AND EVALUATION LOOP ---
    best_val_f1 = 0.0
    print("\nStarting training... 🚀")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
            optimizer.zero_grad()
            
            text_emb = batch['text_emb'].to(DEVICE)
            img_emb = batch['img_emb'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            
            logits = model(text_emb, img_emb)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            
            loss.backward()
            optimizer.step()
        
        avg_train_loss = total_loss / len(train_loader)
        
        val_metrics = evaluate(model, val_loader, DEVICE)
        print(f"Epoch {epoch + 1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val F1: {val_metrics['f1']:.4f} | Val Acc: {val_metrics['accuracy']:.4f}")
        
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"🎉 New best model saved to {BEST_MODEL_PATH} with F1: {best_val_f1:.4f}")

    print("\nTraining finished. ✅")

    # --- FINAL EVALUATION ON TEST SET ---
    print("\nLoading best model and evaluating on the test set...")
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    test_metrics = evaluate(model, test_loader, DEVICE)

    print("\n--- Test Set Results ---")
    print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"F1-Score:  {test_metrics['f1']:.4f}")
    print(f"Precision: {test_metrics['precision']:.4f}")
    print(f"Recall:    {test_metrics['recall']:.4f}")
    print("------------------------")


# --- NEW: The __main__ guard to protect the script's entry point ---
if __name__ == '__main__':
    main()

Using device: cuda
Loaded all text data and image embeddings. ✅
Loading fine-tuned text model for embedding extraction...


Generating text embeddings: 100%|██████████████████████████████████████████████████████| 29/29 [00:40<00:00,  1.40s/it]


Generated text embeddings: Train shape (8400, 1024), Val shape (1800, 1024)

Fusion model created: ✅
FusionModel(
  (fusion_layer): Linear(in_features=1536, out_features=512, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=512, out_features=3, bias=True)
  (relu): ReLU()
)

Starting training... 🚀


Epoch 1/10: 100%|███████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 511.38it/s]


Epoch 1/10 | Train Loss: 1.0770 | Val F1: 0.4952 | Val Acc: 0.4941
🎉 New best model saved to best_fusion_model.pth with F1: 0.4952


...

Epoch 8/10 | Train Loss: 0.7981 | Val F1: 0.5339 | Val Acc: 0.5489


Epoch 9/10: 100%|███████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 756.67it/s]


Epoch 9/10 | Train Loss: 0.7767 | Val F1: 0.5336 | Val Acc: 0.5326


Epoch 10/10: 100%|██████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 772.83it/s]


Epoch 10/10 | Train Loss: 0.7550 | Val F1: 0.5490 | Val Acc: 0.5526
🎉 New best model saved to best_fusion_model.pth with F1: 0.5490

Training finished. ✅

Loading best model and evaluating on the test set...

--- Test Set Results ---
Accuracy:  0.5541
F1-Score:  0.5510
Precision: 0.5511
Recall:    0.5541
------------------------
